# Excel reports through ZEMI Arsenal and GBNF

`MarkItDown` converts three Excel reports to Markdown; llama.cpp applies a GBNF grammar during inference, and Pydantic validates the resulting JSON.

## Input parameters

In [ ]:
arsenal_config_path = "@comp/zemi/llm_curated_set_model_mode.toml"
arsenal_stop_before_playbook_begin = True
arsenal_stop_after_playbook_end = True
model_name = "ling30_tiny"

## Playbook preparation

In [ ]:
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

In [ ]:
import zemi
from zemi.arsenal import ArsenalSession

arsenal = ArsenalSession(arsenal_config_path)
zemi.arsenal.begin(arsenal, stop_before_begin=arsenal_stop_before_playbook_begin)
assistant = arsenal.model(model_name).assistants.assistant
client = assistant.clients.openai.client.with_options(timeout=300.0, max_retries=0)

## Schema and grammar

In [ ]:
from pydantic import BaseModel, ConfigDict, Field

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class Transaction(StrictModel):
    date: str = Field(description="Date in YYYY-MM-DD format")
    article: str
    cost: float

class Report(StrictModel):
    source_file: str
    city: str
    export_date: str = Field(description="Date in YYYY-MM-DD format")
    manager: str
    transactions: list[Transaction]

class Reports(StrictModel):
    reports: list[Report]

grammar = r'''
root ::= ws "{" ws "\"reports\"" ws ":" ws "[" ws report ws "," ws report ws "," ws report ws "]" ws "}" ws
report ::= "{" ws "\"source_file\"" ws ":" ws string ws "," ws "\"city\"" ws ":" ws string ws "," ws "\"export_date\"" ws ":" ws string ws "," ws "\"manager\"" ws ":" ws string ws "," ws "\"transactions\"" ws ":" ws "[" ws transaction (ws "," ws transaction)* ws "]" ws "}"
transaction ::= "{" ws "\"date\"" ws ":" ws string ws "," ws "\"article\"" ws ":" ws string ws "," ws "\"cost\"" ws ":" ws number ws "}"
string ::= "\"" char* "\""
char ::= [^"\\\x7F\x00-\x1F] | "\\" (["\\/bfnrt] | "u" [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F])
number ::= "-"? ("0" | [1-9] [0-9]*) ("." [0-9]+)? ([eE] [+-]? [0-9]+)?
ws ::= [ \t\n\r]?
'''

## Excel to Markdown

In [ ]:
from markitdown import MarkItDown
from zemi import env

data_dir = env.path.comp.root / "data/case01"
excel_files = [data_dir / f"Report {number}.xlsx" for number in range(1, 4)]
if not all(path.is_file() for path in excel_files):
    excel_files = [data_dir / f"Отчет {number}.xlsx" for number in range(1, 4)]
converter = MarkItDown(enable_plugins=False)
excel_context = "\n\n".join(
    f"# File: {path.name}\n\n{converter.convert(path).text_content.strip()}"
    for path in excel_files
)
[(path.name, path.stat().st_size) for path in excel_files]

## GBNF-constrained extraction

In [ ]:
import json

task = """
Extract every report: source file, city/branch, export date, manager, and
transactions (date, article, cost). Omit Total rows, invent nothing, write
all dates as YYYY-MM-DD, and return compact JSON without indentation.
""".strip()

response = client.chat.completions.create(
    model=assistant.clients.model,  # ling-3.0-tiny
    messages=[
        {"role": "system", "content": "Convert Excel reports into strictly structured JSON."},
        {"role": "user", "content": f"{task}\n\n{excel_context}"},
    ],
    temperature=0.0,
    max_tokens=2800,
    extra_body={"grammar": grammar},
)
result = Reports.model_validate_json(response.choices[0].message.content)
print(json.dumps(result.model_dump(mode="json"), ensure_ascii=False, indent=2))

In [ ]:
result

## Stop Arsenal

Run this cell when inference is complete.

In [ ]:
zemi.arsenal.end(arsenal, stop_after_end=arsenal_stop_after_playbook_end)